In [1]:
import os
import sys
from pathlib import Path
import subprocess
from scripts.run_varied_pilot_weekend import collect_corpus, parser
from scripts.weekend_training_progress import CampaignProgress, timing_windows
import time
import signal
import json

C:\Users\theon\.conda\envs\trajectory-nn-clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
JOB = Path("C:/Users/theon/Trajectory_Classification/artifacts/rl_pilot_simplified/vw3")
SCRIPT = 'C:/Users/theon/Trajectory_Classification/scripts/run_corpus_post_weekend.py'
OUTPUT = 'C:/Users/theon/Trajectory_Classification/shared/corpora/nn-duels-varied-v1/vw3'
SIMPLE_EPOCHS = 200
TACTICAL_EPOCHS = 300
ASSESSMENT_SCENARIOS = 300
FLIGHTS = 10000
FLIGHT_SECONDS = 300.0
EPISODE_SECONDS = 300.0
CORPUS_SEED = 800000000
WORKERS = 12
TORCH_THREADS = 2
PILOT_SCHEDULE = 'paired'
INFERENCE_BACKEND = 'auto'
SHARD_FLIGHTS = 100
BENCHMARK_FLIGHTS = 32
ACCEPT_PYARROW_VERSION_CHANGE = False

In [3]:
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p / "scripts/run_varied_pilot_weekend.py").is_file()), None)
ROOT

WindowsPath('C:/Users/theon/Trajectory_Classification')

In [4]:
ARGS = [sys.executable, "-u", str(SCRIPT), "--job", str(JOB), "--output", str(OUTPUT),
        "--simple-epochs", str(SIMPLE_EPOCHS), "--tactical-epochs", str(TACTICAL_EPOCHS),
        "--assessment-scenarios", str(ASSESSMENT_SCENARIOS), "--flights", str(FLIGHTS),
        "--flight-seconds", str(FLIGHT_SECONDS), "--episode-duration-s", str(EPISODE_SECONDS),
        "--evaluation-interval", "1", "--corpus-seed", str(CORPUS_SEED),
        "--workers", str(WORKERS), "--torch-threads", str(TORCH_THREADS),
        "--pilot-schedule", PILOT_SCHEDULE,
        "--inference-backend", INFERENCE_BACKEND, "--shard-flights", str(SHARD_FLIGHTS),
        "--benchmark-flights", str(BENCHMARK_FLIGHTS)]
if ACCEPT_PYARROW_VERSION_CHANGE:
    ARGS.append("--accept-pyarrow-version-change")

In [5]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf8"))

def run_logged(arguments, label):
    logs = JOB / "logs"
    logs.mkdir(exist_ok=True)
    log = logs / f"{label}_{time.time_ns()}.log"
    environment = os.environ.copy()
    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        environment[key] = "1"
    environment["MLFLOW_DISABLE_AGENT_HINT"] = "1"
    print(f"Starting {label}. Full log: {log}", flush=True)
    progress = CampaignProgress(JOB, FLIGHTS) if label == "campaign" else None
    with log.open("w", encoding="utf8") as stream:
        options = {"start_new_session": True} if os.name != "nt" else {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP}
        process = subprocess.Popen(arguments, cwd=ROOT, env=environment, stdout=stream, stderr=subprocess.STDOUT, **options)
        last_report = time.monotonic()
        try:
            while process.poll() is None:
                time.sleep(2)
                if progress is not None:
                    progress.refresh()
                elif time.monotonic() - last_report >= 30:
                    print(f"{label}: checking libraries and saved models...", flush=True)
                    last_report = time.monotonic()
        except BaseException:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], check=False, capture_output=True)
            else:
                os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            if progress is not None:
                progress.close()
    if process.returncode:
        with log.open("rb") as reader:
            reader.seek(max(0, log.stat().st_size - 6000))
            tail = reader.read().decode("utf8", errors="replace")
        raise RuntimeError(f"{label} exited with {process.returncode}. Log: {log}\n{tail}")
    print(f"Completed {label}. Log: {log}", flush=True)
    return log

In [6]:
CAMPAIGN_LOG = run_logged(ARGS, "campaign")
if (JOB / "status.json").exists():
    print(read_json(JOB / "status.json"))

Starting campaign. Full log: C:\Users\theon\Trajectory_Classification\artifacts\rl_pilot_simplified\vw3\logs\campaign_1795184464121906400.log
Overall training: 5000/5000 epochs [elapsed 0:00:00, corpus, training ETA≈0:00:00]
NN-v-NN collection: 4184/10000 flights [elapsed 0:00:00, corpus, collection ETA≈2:05:13]


RuntimeError: campaign exited with 1. Log: C:\Users\theon\Trajectory_Classification\artifacts\rl_pilot_simplified\vw3\logs\campaign_1795184464121906400.log
Traceback (most recent call last):
  File "C:\Users\theon\Trajectory_Classification\scripts\run_corpus_post_weekend.py", line 1116, in <module>
    run(arguments)
  File "C:\Users\theon\Trajectory_Classification\scripts\run_corpus_post_weekend.py", line 1011, in run
    collect_corpus(job, settings)
  File "C:\Users\theon\Trajectory_Classification\scripts\run_corpus_post_weekend.py", line 826, in collect_corpus
    config = duel.prepare(
             ^^^^^^^^^^^^^
  File "C:\Users\theon\Trajectory_Classification\scripts\generate_nn_duel_corpus.py", line 338, in prepare
    raise ValueError(
ValueError: Resume configuration/source/software differs; use a new work directory
